# ResNet-18

Small fake-data smoke run of the natural-instability monitor. This keeps the notebook self-contained and executable on CPU while showing the same control flow as the full experiment.

In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (base / "experiments").exists():
            return base
    raise RuntimeError("Run this notebook from inside the ghosts-of-softmax repository.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_module(name, relative_path):
    path = REPO / relative_path
    module_dir = str(path.parent)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


OUTPUT_ROOT = Path('/tmp/ghosts-of-softmax-notebooks')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO}")
print(f"Notebook outputs: {OUTPUT_ROOT}")


In [ ]:
from IPython.display import Image, display
import torch

resnet = load_module('fig_resnet_run', 'experiments/resnetnatural/run.py')

device = torch.device('cpu')
data_root = OUTPUT_ROOT / 'resnet-data'
data_root.mkdir(parents=True, exist_ok=True)
lrs = [0.02, 0.2]
all_data = {}
for lr in lrs:
    all_data[lr] = resnet.run_training(
        lr=lr,
        epochs=2,
        batch_size=64,
        device=device,
        data_root=data_root,
        seed=0,
        log_every=1,
        dataset='fake',
        train_samples=128,
        test_samples=64,
        num_workers=0,
    )

out_dir = OUTPUT_ROOT / 'resnet18'
out_dir.mkdir(parents=True, exist_ok=True)
out_png = out_dir / 'resnet18-comparison.png'
out_pdf = out_dir / 'resnet18-comparison.pdf'
resnet.make_comparison_plot(all_data, out_png, out_pdf)
display(Image(filename=str(out_png)))

for lr, payload in all_data.items():
    max_r = max(log['r'] for log in payload['step_logs'])
    final_acc = payload['epoch_logs'][-1]['test_acc']
    print(f"lr={lr:.3f} final_acc={final_acc:.3f} max_r={max_r:.3f}")
